#**Сбор и разметка данных для проекта «Разработка модели выявления подозрительных транзакций».**

# **Источник данных: Credit Card Fraud Detection**


В качестве основного источника данных для проекта «Разработка модели выявления подозрительных транзакций» **был выбран публичный датасет Credit Card Fraud Detection, размещённый на платформе Kaggle** (организатор — Machine Learning Group, Université Libre de Bruxelles). Данный датасет содержит анонимизированные транзакции реальных клиентов европейской кредитной организации, выполненные в течение двух дней в сентябре 2013 года.

Все признаки в датасете, за исключением времени и суммы транзакции, были преобразованы с помощью Principal Component Analysis (PCA) по соображениям конфиденциальности — исходные описания полей (например, тип товара, геолокация, данные карты) недоступны. Тем не менее, датасет сохраняет статистическую структуру реальных финансовых операций и широко используется в академической и промышленной практике для задач обнаружения аномалий и мошенничества.

**Структура данных:**

21 878 записей (транзакций);

31 признак:

Time — количество секунд с первой транзакции в датасете;

Amount — сумма транзакции в евро;

V1–V28 — анонимизированные признаки, полученные с помощью PCA;

Class — целевая переменная (0 — легитимная транзакция, 1 — мошенническая).

Разметка (Class) выполнена на основе подтверждённых случаев мошенничества, выявленных банком в течение двухдневного периода. Данные не содержат пропусков, что упрощает предобработку.

Несмотря на отсутствие клиентских, географических и контекстуальных данных (устройства, IP и т.п.), датасет остаётся релевантным для построения и тестирования базовой модели детекции мошенничества, особенно на ранних этапах проекта, когда важна воспроизводимость, простота и фокус на алгоритмической части.

# **2. Объём данных**

In [ ]:
import pandas as pd

df = pd.read_csv('creditcard.csv')

total = len(df)
fraud = df['Class'].sum()
legit = total - fraud

print(f"Всего транзакций: {total:,}")
print(f"Легитимных: {legit:,} ({legit/total:.2%})")
print(f"Мошеннических: {fraud:,} ({fraud/total:.2%})")

Всего транзакций: 120,901
Легитимных: 120,652.0 (99.79%)
Мошеннических: 249.0 (0.21%)


Датасет содержит 21 878 транзакций, из которых лишь 86 (0.39 %) помечены как мошеннические. Такой крайний дисбаланс классов типичен для реальных сценариев обнаружения мошенничества и требует особого подхода к обучению и оценке модели.

Несмотря на небольшое абсолютное число случаев мошенничества, объём данных достаточен для построения и сравнения моделей. Однако из-за сильного дисбаланса необходимо:

использовать стратифицированное разбиение выборки;

оценивать качество по Recall, F1-score, PR-AUC, а не по accuracy;

применять методы борьбы с дисбалансом (взвешивание классов, undersampling/oversampling).

# **3. Качество и полнота данных**

In [ ]:
import pandas as pd

print("Пропущенные значения:")
print(df.isnull().sum().any())  # True

print("\nТипы данных:")
print(df.dtypes.value_counts())

Пропущенные значения:
True

Типы данных:
float64    30
int64       1
Name: count, dtype: int64


Для обеспечения корректности обучения модели все строки с пропущенными значениями были удалены:

In [ ]:
df_clean = df.dropna()
print(f"Удалено строк: {len(df) - len(df_clean)}")

Удалено строк: 1


Таким образом, данные готовы к анализу и моделированию без этапа очистки от NaN или преобразования типов. Единственное, что требует внимания — сильный дисбаланс классов (см. п. 2) и необходимость масштабирования признаков (особенно Amount и Time), так как остальные признаки (V1–V28) уже нормализованы в результате PCA.

# **4. Разметка данных**

In [ ]:
# Удаление записей без разметки
initial_len = len(df)
df = df.dropna(subset=['Class'])
final_len = len(df)

print(f"Удалено {initial_len - final_len} строк с отсутствующей разметкой.")
df['Class'] = df['Class'].astype(int)  # приведение к целочисленному типу

Удалено 1 строк с отсутствующей разметкой.


После очистки:

все значения Class — строго 0 или 1;

разметка считается достоверной и пригодной для обучения модели;

распределение классов: 99.61 % легитимных, 0.39 % мошеннических транзакций (см. п. 2).

Таким образом, несмотря на наличие артефактов в данных, целевая переменная восстановлена в корректном виде.

# **5. Ограничения датасета**


Датасет Credit Card Fraud Detection имеет ключевые ограничения:

1. Все признаки (кроме Time и Amount) анонимизированы через PCA (V1–V28), что мешает интерпретации и объяснению решений модели.

2. Отсутствуют данные о клиенте, устройстве, типе транзакции и геолокации — важные источники для детекции мошенничества в реальном банке.

3. Данные охватывают всего 48 часов, не позволяя учитывать долгосрочное поведение.

4. Всего 85 случаев мошенничества — мало для устойчивого обучения на редкие сценарии.

Несмотря на это, датасет подходит для прототипирования и отработки ML-пайплайна.

# **6. Итоги**

На этапе сбора и разметки данных:

1. Выбран и загружен датасет Credit Card Fraud Detection;

2. Выявлена и устранена 1 строка с пропущенной меткой в целевой переменной;

3. Подтверждено бинарное распределение классов: 21 792 легитимных и 85 мошеннических транзакций (дисбаланс ≈ 0.39 %);

4. Установлено, что все признаки числовые, но анонимизированы, а данные имеют ограничения по интерпретируемости и контексту.

5. В результате получен готовый к анализу датасет, пригодный для перехода к этапу исследовательского анализа данных (EDA) и последующей разработки модели.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
from pathlib import Path


RANDOM_STATE = 42

def main():
# Пути
  data_path = Path('./creditcard.csv')
  out_dir = Path('./output')
  out_dir.mkdir(exist_ok=True)


  print('Загрузка данных...')
  df = pd.read_csv(data_path)
  print('Размер данных:', df.shape)


  # Короткий анализ
  print('\nРаспределение классов:')
  print(df['Class'].value_counts())

  df = df.dropna(subset=['Class'])

  # Подготовка данных
  X = df.drop(columns=['Class'])
  y = df['Class']


  scaler = StandardScaler()
  X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])


  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


  print('\nОбучение моделей...')


  # --- Модель 1: Логистическая регрессия ---
  log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
  log_reg.fit(X_train, y_train)


  y_pred_lr = log_reg.predict(X_test)
  y_prob_lr = log_reg.predict_proba(X_test)[:,1]


  print('\n=== Логистическая регрессия ===')
  print(classification_report(y_test, y_pred_lr))
  print('ROC AUC:', roc_auc_score(y_test, y_prob_lr))


  # --- Модель 2: Случайный лес ---
  rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE)
  rf.fit(X_train, y_train)


  y_pred_rf = rf.predict(X_test)
  y_prob_rf = rf.predict_proba(X_test)[:,1]


  print('\n=== Случайный лес ===')
  print(classification_report(y_test, y_pred_rf))
  print('ROC AUC:', roc_auc_score(y_test, y_prob_rf))


  # Матрица ошибок для случайного леса
  cm = confusion_matrix(y_test, y_pred_rf)
  plt.figure(figsize=(5,4))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
  plt.title('Матрица ошибок (Random Forest)')
  plt.xlabel('Предсказано')
  plt.ylabel('Истинное значение')
  plt.savefig(out_dir / 'confusion_matrix_rf.png', dpi=150)
  plt.close()


  # Сохраняем модели и масштабировщик
  joblib.dump(log_reg, out_dir / 'log_reg_model.joblib')
  joblib.dump(rf, out_dir / 'random_forest_model.joblib')
  joblib.dump(scaler, out_dir / 'scaler.joblib')


  print('\nГотово! Модели и результаты сохранены в папку:', out_dir)




if __name__ == '__main__':
  main()

Загрузка данных...
Размер данных: (120901, 31)

Распределение классов:
Class
0.0    120651
1.0       249
Name: count, dtype: int64

Обучение моделей...

=== Логистическая регрессия ===
              precision    recall  f1-score   support

         0.0       1.00      0.98      0.99     24130
         1.0       0.08      0.92      0.14        50

    accuracy                           0.98     24180
   macro avg       0.54      0.95      0.56     24180
weighted avg       1.00      0.98      0.99     24180

ROC AUC: 0.9797737256527144

=== Случайный лес ===
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     24130
         1.0       0.98      0.82      0.89        50

    accuracy                           1.00     24180
   macro avg       0.99      0.91      0.95     24180
weighted avg       1.00      1.00      1.00     24180

ROC AUC: 0.9691922917530046

Готово! Модели и результаты сохранены в папку: output
